# 06 - Regression - Environmental Anomalies vs Catch Tonnage

Runs semi-log OLS regressions to estimate how temperature and salinity anomalies relate to anchoveta catch tonnage, at three aggregation levels.

**Steps:**
1. Load and clean the enriched catch-environment dataset (notebook 05 output)
2. Build salinity anomaly measures (vs day-of-year climatology and vs 35.1 PSU)
3. Aggregate to daily and weekly totals
4. Run univariate semi-log regressions at cala, daily, and weekly levels
5. Run multivariate specs: combined, interaction, and quadratic terms
6. Compare R2 and semi-elasticities across specs and aggregation levels

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

In [ ]:
data_path = Path("/home/jupyter-daniela/suyana/peru_production/outputs/")
df = pd.read_csv(data_path / "calas_with_hycom_data.csv", parse_dates=['date'])

df_clean = df.dropna(subset=['hycom_temp_anom', 'hycom_sal', 'catch_tm'])
df_clean = df_clean[df_clean['catch_tm'] > 0].copy()

Q1, Q3 = df_clean['catch_tm'].quantile([0.25, 0.75])
IQR = Q3 - Q1
df_clean = df_clean[
    (df_clean['catch_tm'] >= Q1 - 1.5 * IQR) &
    (df_clean['catch_tm'] <= Q3 + 1.5 * IQR)
].copy()

df_clean['doy'] = df_clean['date'].dt.dayofyear
sal_clim = df_clean.groupby('doy')['hycom_sal'].mean()
df_clean['sal_anom'] = df_clean['hycom_sal'] - df_clean['doy'].map(sal_clim)

df_clean['sal_anom_ref35'] = df_clean['hycom_sal'] - 35.1

df_clean['log_catch'] = np.log(df_clean['catch_tm'])

has_modis = 'modis_sst_anom' in df_clean.columns
print(f"N calas: {len(df_clean):,}")
print(f"MODIS SST anomaly available: {has_modis}")
print(df_clean[['hycom_temp_anom', 'sal_anom', 'sal_anom_ref35', 'catch_tm', 'log_catch']].describe().round(3))


## Aggregations

Aggregate individual calas to daily and weekly totals.  
Environmental variables are averaged across all fishing events within each period.

In [ ]:
agg_cols = {
    'catch_tm':      ('catch_tm', 'sum'),
    'temp_anom':     ('hycom_temp_anom', 'mean'),
    'sal_anom':      ('sal_anom', 'mean'),
    'sal_anom_ref35':('sal_anom_ref35', 'mean'),
}
if has_modis:
    agg_cols['modis_sst_anom'] = ('modis_sst_anom', 'mean')

daily = df_clean.groupby('date').agg(**agg_cols).reset_index()
daily['log_catch'] = np.log(daily['catch_tm'])

print(f"Daily observations: {len(daily)}")
print(daily[['catch_tm', 'temp_anom', 'sal_anom', 'sal_anom_ref35']].describe().round(3))


In [ ]:
iso = df_clean['date'].dt.isocalendar()
df_clean['year_week'] = iso.year.astype(str) + '-W' + iso.week.astype(str).str.zfill(2)

weekly = df_clean.groupby('year_week').agg(**agg_cols).reset_index()
weekly['log_catch'] = np.log(weekly['catch_tm'])

print(f"Weekly observations: {len(weekly)}")
print(weekly[['catch_tm', 'temp_anom', 'sal_anom', 'sal_anom_ref35']].describe().round(3))


## Semi-log regressions

**Model**: `log(tns) ~ anomaly`

Coefficients are semi-elasticities:
- β × 100 = % change in catch tonnage per 1-unit increase in anomaly
- Separate regressions for temperature anomaly and salinity anomaly
- Run at three levels: individual cala, daily, weekly

In [ ]:
def reg_table(data, regressors, label):
    """Semi-log OLS. Returns a DataFrame of semi-elasticities."""
    rows = []
    for var in regressors:
        if var not in data.columns:
            continue
        m = smf.ols(f'log_catch ~ {var}', data=data.dropna(subset=[var])).fit()
        beta = m.params[var]
        pval = m.pvalues[var]
        ci_lo, ci_hi = m.conf_int().loc[var]
        rows.append({
            'variable': var,
            'beta': round(beta, 4),
            '% change per unit': round(beta * 100, 2),
            '95% CI (%)': f"[{ci_lo*100:.2f}, {ci_hi*100:.2f}]",
            'p-value': round(pval, 4),
            'R²': round(m.rsquared, 4),
            'N': int(m.nobs),
        })
    result = pd.DataFrame(rows).set_index('variable')
    print(f"\n{'='*65}\n{label}\n{'='*65}")
    print(result.to_string())
    return result

regressors = ['temp_anom', 'sal_anom', 'sal_anom_ref35']
if has_modis:
    regressors += ['modis_sst_anom']

df_ind = df_clean.rename(columns={'hycom_temp_anom': 'temp_anom'})
ind = reg_table(df_ind, regressors, 'Individual cala (Lagrangian — each fishing event)')


In [ ]:
daily_res = reg_table(daily, regressors, 'Daily aggregation')

In [ ]:
weekly_res = reg_table(weekly, regressors, 'Weekly aggregation')

comparison = pd.concat([
    ind[['% change per unit', 'p-value', 'R²']].add_suffix(' (cala)'),
    daily_res[['% change per unit', 'p-value', 'R²']].add_suffix(' (daily)'),
    weekly_res[['% change per unit', 'p-value', 'R²']].add_suffix(' (weekly)'),
], axis=1)

print("\n\n=== SUMMARY: % change in catches per 1-unit anomaly ===")
print(comparison.to_string())

## Multivariate specifications

Three additional models comparing temperature and salinity jointly:

1. **Combined** - `log(tns) ~ temp + sal`  
2. **Interaction** - `log(tns) ~ temp + sal + temp×sal`  
3. **Quadratic** - `log(tns) ~ temp + temp² + sal + sal²`

Run at three levels: individual cala, daily, weekly.  
Uses `temp_anomaly` (HYCOM) and `sal_anomaly` (vs climatology). Where available, also run with `modis_sst_anomaly`.

In [ ]:
def multi_reg(data, formula, label, level):
    """Run OLS and return a summary row + print coefficients."""
    subset = data.dropna(subset=['log_catch']).copy()
    vars_in = [v.strip() for v in formula.split('~')[1].replace('*', '+').replace('I(', '').replace('**2)', '').split('+')]
    vars_in = [v.strip() for v in vars_in if v.strip() in subset.columns]
    subset = subset.dropna(subset=vars_in)
    if len(subset) < 10:
        print(f"  {label} [{level}]: insufficient data")
        return None
    m = smf.ols(formula, data=subset).fit()
    coefs = pd.DataFrame({'coef': m.params, 'p-value': m.pvalues}).round(4)
    coefs = coefs[coefs.index != 'Intercept']
    print(f"\n--- {label} [{level}]  R²={m.rsquared:.4f}  N={int(m.nobs)} ---")
    print(coefs.to_string())
    return {'label': label, 'level': level, 'R2': round(m.rsquared, 4), 'N': int(m.nobs)}

datasets = {
    'cala':   df_clean.rename(columns={'hycom_temp_anom': 'temp_anom'}) if 'hycom_temp_anom' in df_clean.columns else df_clean,
    'daily':  daily,
    'weekly': weekly,
}

specs = {
    'combined':    'log_catch ~ temp_anom + sal_anom',
    'interaction': 'log_catch ~ temp_anom * sal_anom',
    'quadratic':   'log_catch ~ temp_anom + I(temp_anom**2) + sal_anom + I(sal_anom**2)',
}

summary_rows = []
for spec_name, formula in specs.items():
    print(f"\n{'#'*65}\nSPEC: {spec_name}\n{'#'*65}")
    for level, data in datasets.items():
        row = multi_reg(data, formula, spec_name, level)
        if row:
            summary_rows.append(row)


In [ ]:
if has_modis:
    specs_modis = {
        'combined (MODIS)':    'log_catch ~ modis_sst_anom + sal_anom',
        'interaction (MODIS)': 'log_catch ~ modis_sst_anom * sal_anom',
        'quadratic (MODIS)':   'log_catch ~ modis_sst_anom + I(modis_sst_anom**2) + sal_anom + I(sal_anom**2)',
    }
    for spec_name, formula in specs_modis.items():
        print(f"\n{'#'*65}\nSPEC: {spec_name}\n{'#'*65}")
        for level, data in datasets.items():
            row = multi_reg(data, formula, spec_name, level)
            if row:
                summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).pivot(index='label', columns='level', values='R2')[['cala', 'daily', 'weekly']]
print("\n\n=== R² COMPARISON ACROSS SPECS AND LEVELS ===")
print(df_summary.to_string())
